# METHOD V3 — SELECTION → LEARNING TRANSFER
## OVERNIGHT ONE-CLICK EXPERIMENT

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gubiczam/owod-active/blob/main/notebooks/method_v3_selection_transfer_overnight.ipynb)

1. Open in Google Colab
2. Select **T4 GPU**
3. **Runtime → Run all**
4. Come back tomorrow

Google Drive authorization is the only expected interaction. Nothing else needs
touching, and a lost session costs nothing: every trajectory is resumable and a
second **Run all** skips what already finished.

---

**Exploratory / prospective.** Method V2 Stage 2 stands unchanged —
`D_NO_GO`, `R_NO_GO`, `C_GO`, allowed ladder `U`. No threshold is reopened here.
This notebook asks a *new* question: does the one component that passed its own
frozen gate, consistency `C`, produce a downstream active-learning benefit at an
equal annotation budget?

The protocol, the four arms, the seeds, the budget and the success criterion are
frozen in [`docs/method_v3_protocol_2026-09-02.md`](https://github.com/gubiczam/owod-active/blob/main/docs/method_v3_protocol_2026-09-02.md),
written before any downstream detector endpoint existed. This notebook is thin:
it bootstraps, validates, launches and summarises. Every scientific decision
lives in `owl/method_v3.py` and in that document, and **cell [7/9] prints the
success criterion before the training launcher in [8/9] can run.**

| | |
|---|---|
| arms | `random`, `A`, `U`, `A*C` |
| seeds | 0, 1, 2 |
| budget | 600 regions in 6 rounds of 100 |
| trajectories | 12, all attempted |
| replay | `uniform`, 400 exemplar objects, identical for every arm |
| expected T4 time | ≈ 8 hours |


In [ ]:
# [1/9] Parameters and immutable experiment identity
# ============================== PARAMETERS ==============================
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

OWL_REPOSITORY = "https://github.com/gubiczam/owod-active.git"
OWL_COMMIT = "761b7745998fbbeb6d1a363121d09254c2b50cd6"
PROB_REPOSITORY = "https://github.com/gubiczam/PROB.git"
PROB_COMMIT = "4c66be1a52cad9360e09c729e9134aba8fe0b531"

DRIVE_ROOT = "/content/drive/MyDrive/OWL"
CHECKPOINT_RELATIVE = "checkpoints/SOWODB/t1.pth"
FEATURES_RELATIVE = "features"
RESULTS_RELATIVE = "results/method_v3_selection_transfer"

DATA_ROOT = "/content/data/OWOD"

# One Run all is allowed this much GPU time. The launcher stops cleanly between
# trajectories when it is reached and the next Run all resumes; nothing is
# truncated inside a trajectory.
TIME_BUDGET_MINUTES = 660

SESSION_STARTED = time.monotonic()

# Every scientific constant is READ from the pinned repository, never restated
# here — a number typed into a notebook is a number that can drift away from the
# module that the verdict is actually computed from.
assert len(PROB_COMMIT) == 40 and len(OWL_COMMIT) == 40, "pin full 40-char SHAs"
print("OWL commit :", OWL_COMMIT)
print("PROB commit:", PROB_COMMIT)


In [ ]:
# [2/9] Mount Drive and prove the persistent root is writable
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
_probe = DRIVE / ".method_v3_write_probe"
_probe.write_text("ok", encoding="utf-8")
assert _probe.read_text(encoding="utf-8") == "ok"
_probe.unlink()

FEATURES = DRIVE / FEATURES_RELATIVE
RESULTS = DRIVE / RESULTS_RELATIVE
CHECKPOINT = DRIVE / CHECKPOINT_RELATIVE
RESULTS.mkdir(parents=True, exist_ok=True)
print("Drive writable:", DRIVE)
print("results ->", RESULTS)


In [ ]:
# [3/9] Pin OWL exactly, install its declared dependencies, import fresh code
def _checked(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)


def _capture(command, **kwargs):
    return _checked(command, capture_output=True, **kwargs).stdout.strip()


def _normalise_git_url(value):
    value = value.strip().removesuffix(".git").rstrip("/")
    if value.startswith("git@github.com:"):
        value = "https://github.com/" + value.split(":", 1)[1]
    return value


def ensure_pinned_checkout(path, repository, commit):
    path = Path(path)
    expected = _normalise_git_url(repository)
    if path.exists():
        assert (path / ".git").is_dir(), f"Refusing non-git path: {path}"
        origin = _normalise_git_url(_capture(["git", "remote", "get-url", "origin"], cwd=path))
        assert origin == expected, f"Refusing unexpected origin at {path}: {origin}"
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        _checked(["git", "clone", "--filter=blob:none", "--no-checkout", repository, str(path)])
    _checked(["git", "fetch", "--depth", "1", "origin", commit], cwd=path)
    _checked(["git", "reset", "--hard", commit], cwd=path)
    _checked(["git", "clean", "-fdx"], cwd=path)
    actual = _capture(["git", "rev-parse", "HEAD"], cwd=path)
    assert actual == commit, f"{path}: expected {commit}, got {actual}"
    return path


ROOT = ensure_pinned_checkout(Path("/content/owod-active"), OWL_REPOSITORY, OWL_COMMIT)
_checked([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q",
          "-e", f"{ROOT}[plots]"])

for _name in [n for n in sys.modules if n == "owl" or n.startswith("owl.")]:
    del sys.modules[_name]
sys.path.insert(0, str(ROOT))

from owl import bridge, discovery, evaluation_subset, exemplars, labelling
from owl import method_v3, metrics, protocol, replay, scoring, selection

_required = {
    "select.precomputed": "precomputed" in __import__("inspect").signature(selection.select).parameters,
    "method_v3.CRITERION": hasattr(method_v3, "CRITERION"),
    "method_v3.evaluate_criterion": hasattr(method_v3, "evaluate_criterion"),
    "method_v3.run_trajectory": hasattr(method_v3, "run_trajectory"),
    "metrics.validate_per_class_ap50": hasattr(metrics, "validate_per_class_ap50"),
    "metrics.unknown_recall_by_group": hasattr(metrics, "unknown_recall_by_group"),
    "discovery.cumulative": hasattr(discovery, "cumulative"),
}
assert all(_required.values()), {k: v for k, v in _required.items() if not v}
assert method_v3.ARMS == ("random", "A", "U", "A*C"), method_v3.ARMS
assert method_v3.SEEDS == (0, 1, 2), method_v3.SEEDS
assert len(method_v3.trajectories()) == 12
print("arms:", method_v3.ARMS, "| seeds:", method_v3.SEEDS,
      "| budget:", method_v3.BUDGET, "in", method_v3.ROUNDS, "rounds")
OWL_SHA = _capture(["git", "rev-parse", "HEAD"], cwd=ROOT)
print("OWL ready:", OWL_SHA, "from", ROOT)


In [ ]:
# [4/9] Pin and validate the reviewed PROB bridge; build the optional CUDA kernel
PROB = ensure_pinned_checkout(Path("/content/PROB"), PROB_REPOSITORY, PROB_COMMIT)

# PROB's 2022 requirements file pins packages that have no Python 3.13 wheels
# (notably scikit-image 0.19.2 and pandas 1.5.1). The bridge does not import
# scikit-image, notebook, or ipdb. Install only its runtime imports, without
# replacing Colab's matched torch/torchvision/numpy stack. pycocotools stays
# at PROB's exact 2.0.5 pin: Cython generates the C source omitted by its sdist.
assert sys.version_info[:2] == (3, 13), sys.version
def distribution_version(distribution):
    probe = subprocess.run(
        [sys.executable, "-c",
         f"from importlib.metadata import version; print(version({distribution!r}))"],
        capture_output=True, text=True, check=False)
    return probe.stdout.strip() if probe.returncode == 0 else None


def module_available(module):
    return subprocess.run(
        [sys.executable, "-c", f"import {module}"],
        capture_output=True, text=True, check=False).returncode == 0


PROB_COMPAT_INSTALLED = []
if distribution_version("einops") != "0.5.0" or not module_available("einops"):
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "einops==0.5.0"])
    PROB_COMPAT_INSTALLED.append("einops==0.5.0")

if (distribution_version("pycocotools") != "2.0.5"
        or not module_available("pycocotools")):
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "Cython==3.1.3"])
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--no-build-isolation",
              "--no-deps", "--force-reinstall", "pycocotools==2.0.5"])
    PROB_COMPAT_INSTALLED.append("pycocotools==2.0.5")

# These imports are required transitively by main_open_world/engine. Keep a
# compatible Colab package when one is already importable; install a pinned
# Python-3.13 wheel only when it is absent or broken. WandB is disabled by the
# reviewed bridge and therefore cannot affect training or evaluation.
compatibility_wheels = {
    "wandb": "wandb==0.18.7",
    "pandas": "pandas==2.3.2",
    "seaborn": "seaborn==0.13.2",
    "tqdm": "tqdm==4.67.1",
}
missing_wheels = [spec for module, spec in compatibility_wheels.items()
                  if not module_available(module)]
if missing_wheels:
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              *missing_wheels])
    PROB_COMPAT_INSTALLED.extend(missing_wheels)

assert distribution_version("einops") == "0.5.0" and module_available("einops")
assert (distribution_version("pycocotools") == "2.0.5"
        and module_available("pycocotools"))


def pip_check():
    return subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        capture_output=True, text=True, check=False)


# Current Colab carries IPython metadata that requires Jedi while omitting
# Jedi itself. Repair only that observed metadata conflict, using a universal
# wheel with explicit Python 3.13 support, then require the complete package
# environment to pass the same check used by the final preflight.
bootstrap_package_probe = pip_check()
_package_conflicts = bootstrap_package_probe.stdout + bootstrap_package_probe.stderr
_missing_ipython_jedi = (
    "requires jedi, which is not installed" in _package_conflicts.lower()
    and "ipython " in _package_conflicts.lower()
)
if _missing_ipython_jedi:
    _checked([sys.executable, "-m", "pip", "install",
              "--disable-pip-version-check", "-q", "--only-binary=:all:",
              "jedi==0.19.2"])
    PROB_COMPAT_INSTALLED.append("jedi==0.19.2")
    bootstrap_package_probe = pip_check()
if bootstrap_package_probe.returncode != 0:
    print(bootstrap_package_probe.stdout + bootstrap_package_probe.stderr)
    raise RuntimeError("Python package consistency check failed after bootstrap repair")
print("Bootstrap package consistency: PASS")
runtime_probe = subprocess.run(
    [sys.executable, "-c",
     "import numpy, torch, torchvision, scipy, sklearn, PIL, matplotlib, pandas, seaborn, tqdm, wandb; "
     "from einops import rearrange; from pycocotools.coco import COCO; "
     "import main_open_world; from datasets.coco import make_coco_transforms; "
     "from datasets.torchvision_datasets.open_world import OWDetection; "
     "from engine import evaluate; from models import build_model"],
    cwd=PROB, capture_output=True, text=True, check=False)
if runtime_probe.returncode != 0:
    print(runtime_probe.stdout)
    print(runtime_probe.stderr)
    raise RuntimeError("Pinned PROB failed its Python runtime import probe")
print("PROB runtime imports: PASS; installed:", PROB_COMPAT_INSTALLED or "nothing")

# pycocotools 2.0.5 predates NumPy 2.0. Exercise PROB's own evaluator
# wrapper with the two removed aliases it needs, instead of accepting an
# import-only success. The aliases are confined to this fresh subprocess.
coco_smoke_code = r'''
import numpy as np
import torch
if "float" not in np.__dict__:
    np.float = float
if "NPY_OWNDATA" not in np.__dict__:
    np.NPY_OWNDATA = 4
from pycocotools.coco import COCO
from datasets.coco_eval import CocoEvaluator
coco = COCO()
coco.dataset = {
    "info": {}, "licenses": [],
    "images": [{"id": 1, "width": 32, "height": 32}],
    "categories": [{"id": 1, "name": "object", "supercategory": "object"}],
    "annotations": [{"id": 1, "image_id": 1, "category_id": 1,
                     "bbox": [4.0, 5.0, 10.0, 11.0], "area": 110.0, "iscrowd": 0}],
}
coco.createIndex()
evaluator = CocoEvaluator(coco, ("bbox",))
evaluator.update({1: {"boxes": torch.tensor([[4.0, 5.0, 14.0, 16.0]]),
                      "scores": torch.tensor([0.99]), "labels": torch.tensor([1])}})
evaluator.synchronize_between_processes()
evaluator.accumulate()
evaluator.summarize()
assert float(evaluator.coco_eval["bbox"].stats[0]) > 0.99
print("PROB pycocotools COCOeval smoke: PASS")
'''
coco_smoke = subprocess.run([sys.executable, "-c", coco_smoke_code], cwd=PROB,
                            capture_output=True, text=True, check=False)
if coco_smoke.returncode != 0:
    print(coco_smoke.stdout)
    print(coco_smoke.stderr)
    raise RuntimeError("Pinned pycocotools failed PROB's functional COCOeval smoke test")
print(coco_smoke.stdout.splitlines()[-1])


def run_json_probe(code, marker, *, cwd):
    probe = subprocess.run(
        [sys.executable, "-c", code], cwd=cwd,
        capture_output=True, text=True, check=False,
    )
    rows = [line.removeprefix(marker) for line in probe.stdout.splitlines()
            if line.startswith(marker)]
    if not rows:
        return {
            "probe_ok": False,
            "returncode": probe.returncode,
            "error": (probe.stderr or probe.stdout).strip() or "probe produced no result",
        }
    payload = json.loads(rows[-1])
    payload["returncode"] = probe.returncode
    return payload


# Deliberately diagnostic only: unlike PROB, this fresh interpreter does not
# import torch before loading the extension. On Colab that can fail to resolve
# PyTorch shared libraries even when PROB's real import and dispatch work.
raw_msda_probe_code = r"""
import importlib
import json
try:
    importlib.invalidate_caches()
    extension = importlib.import_module("MultiScaleDeformableAttention")
    payload = {"ok": True, "path": getattr(extension, "__file__", None), "error": None}
except BaseException as error:
    payload = {"ok": False, "path": None,
               "error": f"{type(error).__name__}: {error}"}
print("OWOD_RAW_MSDA_PROBE=" + json.dumps(payload, sort_keys=True))
"""
RAW_MSDA = run_json_probe(
    raw_msda_probe_code, "OWOD_RAW_MSDA_PROBE=", cwd=PROB.parent)

# Authoritative pre-build/post-build probe: import through the exact wrapper and
# downstream module used by PROB training. The pinned wrapper imports torch
# before the extension and the downstream module copies this boolean for dispatch.
prob_msda_probe_code = r"""
import importlib
import importlib.metadata
import json
import platform
import sys
from pathlib import Path
import einops
import matplotlib
import numpy
import pandas
import PIL
import pycocotools
import scipy
import sklearn
import torch
import torchvision

after_torch = {"ok": False, "path": None, "error": None}
try:
    importlib.invalidate_caches()
    extension = importlib.import_module("MultiScaleDeformableAttention")
    after_torch = {"ok": True, "path": getattr(extension, "__file__", None), "error": None}
except BaseException as error:
    after_torch["error"] = f"{type(error).__name__}: {error}"

payload = {
    "probe_ok": False,
    "python": platform.python_version(),
    "executable": sys.executable,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "torch_cuda": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "numpy": numpy.__version__,
    "scipy": scipy.__version__,
    "sklearn": sklearn.__version__,
    "pillow": PIL.__version__,
    "matplotlib": matplotlib.__version__,
    "pandas": pandas.__version__,
    "einops": importlib.metadata.version("einops"),
    "pycocotools": importlib.metadata.version("pycocotools"),
    "extension_after_torch": after_torch,
}
try:
    from models.ops.functions import ms_deform_attn_func as msda_func
    from models.ops.modules import ms_deform_attn as msda_module
    expected_root = Path.cwd().resolve()
    wrapper_path = Path(msda_func.__file__).resolve()
    downstream_path = Path(msda_module.__file__).resolve()
    assert wrapper_path.is_relative_to(expected_root), wrapper_path
    assert downstream_path.is_relative_to(expected_root), downstream_path
    available = bool(msda_func.MSDA_AVAILABLE)
    assert bool(msda_module.MSDA_AVAILABLE) == available
    payload.update({
        "probe_ok": True,
        "available": available,
        "backend": "compiled" if available else "PyTorch fallback",
        "wrapper_path": str(wrapper_path),
        "downstream_path": str(downstream_path),
        "extension_path": (getattr(msda_func.MSDA, "__file__", None)
                           if available else None),
        "error": None,
    })
except BaseException as error:
    payload["error"] = f"{type(error).__name__}: {error}"
print("OWOD_PROB_MSDA_PROBE=" + json.dumps(payload, sort_keys=True))
"""


def probe_prob_msda():
    return run_json_probe(
        prob_msda_probe_code, "OWOD_PROB_MSDA_PROBE=", cwd=PROB)


PREBUILD_PROB_MSDA = probe_prob_msda()
_msda_fingerprint = json.dumps({
    "prob": PROB_COMMIT,
    "python": PREBUILD_PROB_MSDA.get("python", sys.version),
    "torch": PREBUILD_PROB_MSDA.get("torch"),
    "torch_cuda": PREBUILD_PROB_MSDA.get("torch_cuda"),
    "gpu": PREBUILD_PROB_MSDA.get("gpu"),
}, sort_keys=True)
MSDA_BUILD_MARKER = (
    PROB.parent / ".owod-active-cache" /
    f"msda-build-{hashlib.sha256(_msda_fingerprint.encode()).hexdigest()[:16]}.json"
)
MSDA_BUILD_ATTEMPTED = False
MSDA_BUILD_RETURN_CODE = None
if PREBUILD_PROB_MSDA.get("available") is not True:
    if MSDA_BUILD_MARKER.is_file():
        print("Skipping a previously built-but-unused MSDA extension for this exact runtime:",
              MSDA_BUILD_MARKER)
    else:
        if not module_available("ninja"):
            _checked([sys.executable, "-m", "pip", "install",
                      "--disable-pip-version-check", "-q", "ninja"])
        MSDA_BUILD_ATTEMPTED = True
        build = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--no-build-isolation", "--no-deps", "--force-reinstall", "."],
            cwd=PROB / "models" / "ops", text=True,
            capture_output=True, check=False,
        )
        MSDA_BUILD_RETURN_CODE = build.returncode
        if build.returncode != 0:
            print("WARNING: optional CUDA extension build failed; the full CUDA smoke "
                  "must prove PROB's PyTorch fallback.")
            print("\n".join((build.stdout + "\n" + build.stderr).splitlines()[-20:]))

POSTBUILD_PROB_MSDA = probe_prob_msda()
print("Raw extension import:", "PASS" if RAW_MSDA.get("ok") else "FAIL")
print("Raw extension path:", RAW_MSDA.get("path") or "unavailable")
if RAW_MSDA.get("error"):
    print("Raw extension error:", RAW_MSDA["error"])
print("PROB MSDA_AVAILABLE:", POSTBUILD_PROB_MSDA.get("available"))
print("PROB wrapper path:", POSTBUILD_PROB_MSDA.get("wrapper_path", "unavailable"))
print("PROB extension path:", POSTBUILD_PROB_MSDA.get("extension_path", "unavailable"))
if RAW_MSDA.get("ok") != POSTBUILD_PROB_MSDA.get("available"):
    print("MSDA diagnostic disagreement explained: the raw probe loads the extension "
          "before torch; pinned PROB imports torch first, then binds the extension "
          "inside models.ops.functions.ms_deform_attn_func. Only PROB's downstream "
          "dispatch is authoritative.")

# Run the real PROB builder and training loss on CUDA. This observes the exact
# MSDeformAttn branch taken by the model and requires that branch to participate
# in forward and backward before any experiment evaluation or training can run.
prob_smoke_code = r"""
import json
import sys
import tempfile
from pathlib import Path
import numpy as np
import torch
from PIL import Image
if "bool" not in np.__dict__:
    np.bool = np.bool_
import main_open_world
from datasets.coco import make_coco_transforms
from datasets.open_world_eval import voc_eval
from datasets.torchvision_datasets.open_world import OWDetection
from models import build_model
from models.ops.functions import ms_deform_attn_func as msda_func
from models.ops.modules import ms_deform_attn as msda_module
assert torch.cuda.is_available(), "CUDA is unavailable to the real PROB smoke test"
PROB_MSDA_AVAILABLE = bool(msda_func.MSDA_AVAILABLE)
assert bool(msda_module.MSDA_AVAILABLE) == PROB_MSDA_AVAILABLE
assert Path(msda_func.__file__).resolve().is_relative_to(Path.cwd().resolve())
assert Path(msda_module.__file__).resolve().is_relative_to(Path.cwd().resolve())
dispatch_counts = {"compiled": 0, "fallback": 0}
if PROB_MSDA_AVAILABLE:
    original_apply = msda_module.MSDeformAttnFunction.apply
    class ObservedCompiledDispatch:
        @staticmethod
        def apply(*arguments):
            dispatch_counts["compiled"] += 1
            return original_apply(*arguments)
    msda_module.MSDeformAttnFunction = ObservedCompiledDispatch
else:
    original_fallback = msda_module.ms_deform_attn_core_pytorch
    def observed_fallback(*arguments, **keywords):
        dispatch_counts["fallback"] += 1
        return original_fallback(*arguments, **keywords)
    msda_module.ms_deform_attn_core_pytorch = observed_fallback
args = main_open_world.get_args_parser().parse_args([])
args.device = "cuda"
args.dataset = "OWDETR"
args.PREV_INTRODUCED_CLS = 0
args.CUR_INTRODUCED_CLS = 20
args.num_classes = 81
args.model_type = "prob"
args.wandb_project = ""
args.wandb_name = ""
args.batch_size = 1
args.num_workers = 0
with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    (root / "Annotations").mkdir()
    (root / "JPEGImages").mkdir()
    (root / "ImageSets" / "OWDETR").mkdir(parents=True)
    image_id = "000000000001"
    (root / "ImageSets" / "OWDETR" / "smoke_val.txt").write_text(image_id + "\n")
    Image.new("RGB", (32, 32), (0, 0, 0)).save(root / "JPEGImages" / f"{image_id}.jpg")
    annotation = ("<annotation><filename>000000000001.jpg</filename>"
                  "<size><width>32</width><height>32</height><depth>3</depth></size>"
                  "<object><name>aeroplane</name><difficult>0</difficult>"
                  "<bndbox><xmin>1</xmin><ymin>1</ymin><xmax>20</xmax><ymax>20</ymax>"
                  "</bndbox></object></annotation>")
    xml_path = root / "Annotations" / f"{image_id}.xml"
    xml_path.write_text(annotation)
    dataset = OWDetection(args, root, image_set="smoke_val", dataset="OWDETR",
                          transforms=make_coco_transforms("smoke_val"))
    image, target = dataset[0]
    assert image.shape[0] == 3 and target["labels"].tolist() == [0]
    _, _, ap, *_ = voc_eval(
        [f"{image_id} 0.99 1 1 20 20"], [str(xml_path)], [image_id],
        "aeroplane", known_classes=["aeroplane"])
    assert float(ap) > 0.99
model, criterion, postprocessors, _ = build_model(args, mode="prob")
checkpoint_path = Path(sys.argv[1]) if sys.argv[1] else None
if checkpoint_path is not None:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    assert isinstance(checkpoint, dict) and isinstance(checkpoint.get("epoch"), int)
    state = checkpoint.get("model", checkpoint)
    model_state = model.state_dict()
    compatible_keys = [name for name, value in state.items()
                       if name in model_state and torch.is_tensor(value)
                       and value.shape == model_state[name].shape]
    assert compatible_keys, "T1 checkpoint has no compatible detector parameters"
    incompatible = model.load_state_dict(state, strict=False)
    assert torch.equal(model.state_dict()[compatible_keys[0]].cpu(),
                       state[compatible_keys[0]].cpu())
    print("T1 checkpoint parsed:", checkpoint["epoch"], len(compatible_keys),
          len(incompatible.missing_keys), len(incompatible.unexpected_keys))
model.to("cuda").train()
outputs = model([torch.rand(3, 64, 64, device="cuda")])
required = {"pred_logits", "pred_boxes", "pred_obj", "pred_features", "aux_outputs"}
assert required <= set(outputs)
targets = [{"labels": torch.tensor([0], device="cuda"),
            "boxes": torch.tensor([[0.5, 0.5, 0.25, 0.25]], device="cuda")}]
losses = criterion(outputs, targets)
weighted = sum(losses[name] * criterion.weight_dict[name]
               for name in losses if name in criterion.weight_dict)
assert torch.isfinite(weighted)
weighted.backward()
results = postprocessors["bbox"](outputs, torch.tensor([[64, 64]], device="cuda"))
assert len(results) == 1 and torch.isfinite(results[0]["boxes"]).all()
torch.cuda.synchronize()
chosen = "compiled" if PROB_MSDA_AVAILABLE else "PyTorch fallback"
assert dispatch_counts["compiled" if PROB_MSDA_AVAILABLE else "fallback"] > 0
assert dispatch_counts["fallback" if PROB_MSDA_AVAILABLE else "compiled"] == 0
print("OWOD_MSDA_RESULT=" + json.dumps({
    "available": PROB_MSDA_AVAILABLE,
    "backend": chosen,
    "dispatch_counts": dispatch_counts,
}, sort_keys=True))
print("MSDA backend:", chosen)
print("PROB CUDA model/loss/evaluator smoke: PASS")
"""
checkpoint_for_smoke = Path(DRIVE_ROOT) / CHECKPOINT_RELATIVE
smoke_checkpoint = str(checkpoint_for_smoke) if checkpoint_for_smoke.is_file() else ""
prob_smoke = subprocess.run(
    [sys.executable, "-c", prob_smoke_code, smoke_checkpoint],
    cwd=PROB, capture_output=True, text=True, check=False)
if prob_smoke.returncode != 0:
    print(prob_smoke.stdout)
    print(prob_smoke.stderr)
    print("Environment preflight: FAIL")
    raise RuntimeError("Pinned PROB failed its real CUDA model/evaluator smoke test")
_smoke_rows = [line.removeprefix("OWOD_MSDA_RESULT=")
               for line in prob_smoke.stdout.splitlines()
               if line.startswith("OWOD_MSDA_RESULT=")]
if not _smoke_rows:
    print(prob_smoke.stdout)
    print("Environment preflight: FAIL")
    raise RuntimeError("Real PROB smoke passed without an authoritative MSDA result")
MSDA_SMOKE_RESULT = json.loads(_smoke_rows[-1])
PROB_MSDA_AVAILABLE = bool(MSDA_SMOKE_RESULT["available"])
PROB_MSDA_BACKEND = MSDA_SMOKE_RESULT["backend"]
assert PROB_MSDA_BACKEND == ("compiled" if PROB_MSDA_AVAILABLE else "PyTorch fallback")
if (MSDA_BUILD_ATTEMPTED and MSDA_BUILD_RETURN_CODE == 0
        and not PROB_MSDA_AVAILABLE):
    MSDA_BUILD_MARKER.parent.mkdir(parents=True, exist_ok=True)
    MSDA_BUILD_MARKER.write_text(json.dumps({
        "fingerprint": _msda_fingerprint,
        "backend": PROB_MSDA_BACKEND,
        "reason": "extension built but PROB selected and fully verified fallback",
    }, indent=2), encoding="utf-8")
PROB_SHA = _capture(["git", "rev-parse", "HEAD"], cwd=PROB)
assert PROB_SHA == PROB_COMMIT
ENVIRONMENT_PREFLIGHT_OK = True
runtime = POSTBUILD_PROB_MSDA
print("=" * 60)
print("OWOD ENVIRONMENT PREFLIGHT")
print("=" * 60)
for label, value in (
    ("Runtime Python", runtime.get("python")),
    ("Torch", runtime.get("torch")),
    ("Torchvision", runtime.get("torchvision")),
    ("Torch CUDA", runtime.get("torch_cuda")),
    ("CUDA available", runtime.get("cuda_available")),
    ("GPU", runtime.get("gpu")),
    ("NumPy", runtime.get("numpy")),
    ("SciPy", runtime.get("scipy")),
    ("sklearn", runtime.get("sklearn")),
    ("Pillow", runtime.get("pillow")),
    ("matplotlib", runtime.get("matplotlib")),
    ("pandas", runtime.get("pandas")),
    ("einops", runtime.get("einops")),
    ("pycocotools", runtime.get("pycocotools")),
    ("OWL SHA", OWL_SHA),
    ("PROB SHA", PROB_SHA),
    ("Raw MSDA extension import", "PASS" if RAW_MSDA.get("ok") else "FAIL"),
    ("PROB MSDA_AVAILABLE", PROB_MSDA_AVAILABLE),
    ("MSDA backend", PROB_MSDA_BACKEND),
    ("PROB CUDA model/loss/evaluator smoke", "PASS"),
    ("Environment preflight", "PASS"),
):
    print(f"{label}: {value}")
print("=" * 60)


In [ ]:
# [5/9] The frozen artefacts this experiment reads from Drive — nothing is recomputed
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


BASE_EXPORT = FEATURES / method_v3.BASE_EXPORT_NAME
VIEWS_EXPORT = FEATURES / method_v3.VIEWS_EXPORT_NAME

missing = [str(path) for path in (CHECKPOINT, BASE_EXPORT, VIEWS_EXPORT)
           if not path.is_file()]
assert not missing, (
    "Method V3 reuses completed artefacts and recomputes none of them. Missing "
    f"from Drive: {missing}. The DINOv2 exports are produced by the Method V2 "
    "Stage-2 notebook; t1.pth is PROB's published S-OWODB checkpoint.")

CHECKPOINT_SHA = sha256(CHECKPOINT)
print("checkpoint  :", CHECKPOINT, f"({CHECKPOINT.stat().st_size / 1e6:.0f} MB)")
print("             sha256", CHECKPOINT_SHA)
print("base export :", BASE_EXPORT.name, f"({BASE_EXPORT.stat().st_size / 1e6:.0f} MB)")
print("views export:", VIEWS_EXPORT.name, f"({VIEWS_EXPORT.stat().st_size / 1e6:.0f} MB)")
print()
print("NOT read by Method V3: ref_t1_dinov2_vitb14_cap1000_v1.npz — that is a "
      "Stage-2 reference; C needs the base export and the two views only.")
print("No DINOv2 forward pass runs tonight.")


In [ ]:
# [6/9] Build the data root: committed annotations, the shared split, and the pixels
def _streamed(command, transcript=None):
    """Run a step, showing its output as it happens, and fail loudly.

    ``capture_output=True`` hides the traceback of the step that failed, which is
    the one thing a 3 a.m. Run all must not do. Everything expensive is streamed
    instead, and the exit code is asserted afterwards.
    """

    print("+", " ".join(map(str, command)))
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in process.stdout:
        print(line.rstrip(), flush=True)
        lines.append(line)
    code = process.wait()
    if transcript is not None:
        Path(transcript).parent.mkdir(parents=True, exist_ok=True)
        with Path(transcript).open("a", encoding="utf-8") as handle:
            handle.write(f"\n=== {time.strftime('%Y-%m-%dT%H:%M:%S')} ===\n")
            handle.writelines(lines)
    assert code == 0, f"{command[1]} exited {code}; its output is above"
    return "".join(lines)


_streamed([sys.executable, str(ROOT / "tools" / "prepare_method_v3_data.py"),
           "--data-root", DATA_ROOT])

DATA = Path(DATA_ROOT)
assert (DATA / "Annotations").is_dir() and (DATA / "JPEGImages").is_dir()
assert (DATA / "ImageSets" / "OWDETR" /
        f"{evaluation_subset.SHARED_TEST_SET}.txt").is_file()
print("data root ready:", DATA)


In [ ]:
# [7/9] PREFLIGHT — the runtime estimate and the FROZEN SUCCESS CRITERION,
#       printed before the training launcher in [8/9] can run.
_streamed([sys.executable, str(ROOT / "tools" / "plan_method_v3.py"),
           "--export", str(BASE_EXPORT), "--views", str(VIEWS_EXPORT),
           "--out", str(RESULTS / "runtime_estimate.json")])

# The criterion the verdict is computed from, read out of the pinned module.
CRITERION_STATEMENT = method_v3.CRITERION.statement()
print("=" * 78)
print("FROZEN CRITERION — fixed before execution, not chosen after results")
print("=" * 78)
print(CRITERION_STATEMENT)
print("=" * 78)

# It must already be written down in the protocol document, as the same VALUES.
# Compared field by field against the document's machine-readable block (§11.0),
# never by searching the prose for a rendered phrase: `f"{1.0:g}"` is "1" while
# the document says "1.0", and that difference in formatting once stopped this
# run before it trained anything. A number compared as a number cannot fail that
# way, and a number that really differs cannot pass.
_criterion = method_v3.check_protocol_criterion()
print("criterion frozen in", Path(_criterion["protocol"]).name, "— PASS")
for _field, _value in _criterion["criterion"].items():
    print(f"    {_field:26s} {_value!r}")

# Everything the launcher will be told to do, checked against its own argparse.
_launcher = ROOT / "tools" / "run_method_v3.py"
_help = _capture([sys.executable, str(_launcher), "--help"])
for _flag in ("--prob-root", "--data-root", "--checkpoint", "--export", "--views",
              "--out", "--time-budget-minutes"):
    assert _flag in _help, f"{_launcher.name} has no {_flag}"
print("launcher flags — PASS")

print()
print("The full 12-trajectory orchestration, its resume behaviour and the "
      "verdict machinery are proven without a GPU by tests/test_method_v3.py "
      "(tools/run_method_v3.py --dry-run stubs PROB out). Nothing below reuses a "
      "stubbed trajectory: dry_run is part of the trajectory fingerprint.")


In [ ]:
# [8/9] Run or resume all twelve trajectories. This is the overnight cell.
#
# 4 arms x 3 seeds, in a fixed order, every one attempted. Each trajectory writes
# its own result.json, status.json, selection_curve.csv and checkpoint under
# Drive, so a lost Colab session costs at most the trajectory it was inside.
# Press Run all again and it resumes; nothing complete is repeated.
_launch = [
    sys.executable, str(ROOT / "tools" / "run_method_v3.py"),
    "--prob-root", str(PROB),
    "--data-root", DATA_ROOT,
    "--checkpoint", str(CHECKPOINT),
    "--export", str(BASE_EXPORT),
    "--views", str(VIEWS_EXPORT),
    "--out", str(RESULTS),
    "--time-budget-minutes", str(TIME_BUDGET_MINUTES),
]
# A failure here leaves the failing trajectory's status.json saying FAILED; it is
# never counted as complete. Read it, fix the cause, and Run all again — every
# finished trajectory is kept.
_streamed(_launch, transcript=RESULTS / "run_log.txt")

MANIFEST = json.loads((RESULTS / "manifest.json").read_text(encoding="utf-8"))
COMPLETE = [name for name, state in MANIFEST["trajectory_status"].items()
            if state == method_v3.STATUS_COMPLETE]
print()
print(f"complete: {len(COMPLETE)} of 12")
assert MANIFEST["dry_run"] is False, "this manifest came from a stubbed run"
assert len(COMPLETE) == 12, (
    f"{12 - len(COMPLETE)} trajectories still to run: "
    f"{sorted(set(MANIFEST['trajectory_status']) - set(COMPLETE))}. "
    "Run all again to resume — the verdict is deliberately not computed from an "
    "incomplete design.")


In [ ]:
# [9/9] The tables, the curves and the mechanically generated verdict
_report = _streamed(
    [sys.executable, str(ROOT / "tools" / "summarize_method_v3.py"),
     "--results", str(RESULTS)])
(RESULTS / "method_v3_report.txt").write_text(_report, encoding="utf-8")

SUMMARY = json.loads((RESULTS / "method_v3_summary.json").read_text(encoding="utf-8"))
assert SUMMARY["dry_run"] is False
assert SUMMARY["trajectories_complete"] == SUMMARY["trajectories_expected"] == 12
assert SUMMARY["verdict"] in ("C_DOWNSTREAM_POSITIVE", "C_DOWNSTREAM_NOT_SUPPORTED")

print()
print("=" * 78)
print("METHOD V3 COMPLETE")
print("=" * 78)
print("OWL SHA  :", OWL_SHA)
print("PROB SHA :", PROB_SHA)
print("GPU      :", POSTBUILD_PROB_MSDA.get("gpu"), "| MSDA:", PROB_MSDA_BACKEND)
print("checkpoint sha256:", CHECKPOINT_SHA)
print("arms     :", MANIFEST["arms"], "| seeds:", MANIFEST["seeds"])
print("budget   :", MANIFEST["budget"], "regions in", MANIFEST["rounds"], "rounds")
print("replay   :", MANIFEST["replay"])
print("test set :", MANIFEST["evaluation"]["test_set"],
      f"({MANIFEST['evaluation']['images']} images)")
print("results  :", RESULTS)
print("session elapsed minutes:",
      round((time.monotonic() - SESSION_STARTED) / 60.0, 1))
print()
print("Method V2 Stage 2 is unchanged: D_NO_GO, R_NO_GO, C_GO, allowed ladder U.")
print("VERDICT:", SUMMARY["verdict"])
print("=" * 78)
